# **Pull GitHub Repository**

In [ ]:
!pip install -q torchmetrics timm

In [ ]:
from google.colab import userdata

!rm -rf /content/ECM3401_Individual_Project

token = userdata.get('GitHub')
!git clone -b swin_transformer_layer https://{token}@github.com/sccthomas/ECM3401_Individual_Project.git

In [ ]:
import sys

sys.path.append('/content/ECM3401_Individual_Project/')
!ls /content/ECM3401_Individual_Project/src/

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

# **Define the Model**

In [ ]:
import torch
from src.vision_transformer.model import SemanticSegmentationVisionTransformer

# --------------------------------------------
# Parameters
# --------------------------------------------

device = torch.device("cuda")
metric_device = torch.device("cpu")

image_dims = (3, 512, 512)  # Input image dimensions
patch_embedding_scale_1 = (32, 1024)  # Patch size and embedding dimension for scale 1
patch_embedding_scale_2 = (16, 768)  # Patch size and embedding dimension for scale 2
patch_embedding_scale_3 = (8, 512)  # Patch size and embedding dimension for scale 3

# --------------------------------------------
# Model Initialization
# --------------------------------------------
model = SemanticSegmentationVisionTransformer(
    # - Image dimensions
    image_dims=image_dims,
    # - Hyper Parameters
    num_encoder_layers=6,
    use_heavyweight_decoder=False,
    use_swin_transformer=True,
    use_skip_layer_gated_attention=False,
    skip_layer_ratio=2,
    encoder_dropout_rate=0.15,
    patch_fusion_dropout_rate=0.25,
    decoder_dropout_rate=0.25,
    num_encoder_heads=4,
    num_classes=1,
    # - Scales
    patch_embedding_scale_1=patch_embedding_scale_1,
    patch_embedding_scale_2=patch_embedding_scale_2,
    patch_embedding_scale_3=patch_embedding_scale_3,
).to(device)

# **Train the Model**

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from src.dataset.snow import SnowDataset
from src.training.train import train_model

import os as _os

_os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
# --------------------------------------------
# Parameters
# --------------------------------------------
dataset_dir = "/content/drive/MyDrive/snow_dataset"  # Replace with your dataset path
batch_size = 16
num_epochs = 10
learning_rate = 1e-4
patience = 5  # Early stopping patience

# --------------------------------------------
# Dataset and DataLoader
# --------------------------------------------

# Load the dataset and split it into train and validation sets
train_dataset = SnowDataset(
    dataset_dir_path=dataset_dir,
    len_override=10_000,
    normalize=True,
)
print("Length of dataset:", len(train_dataset))
train_size = int(0.8 * len(train_dataset))
val_size = len(train_dataset) - train_size
train_dataset, val_dataset = random_split(train_dataset, [train_size, val_size])

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=12,
    persistent_workers=True,
    pin_memory=True,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=12,
    persistent_workers=True,
    pin_memory=True,
)

# --------------------------------------------
# Loss, Optimizer, and Scheduler
# --------------------------------------------
criterion = nn.BCEWithLogitsLoss()  # Binary cross-entropy loss with logits
optimizer = optim.Adam(model.parameters(), lr=learning_rate)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)  # Use CosineAnnealingLR scheduler

# --------------------------------------------
# Mixed Precision Setup
# --------------------------------------------
scaler = torch.amp.GradScaler(device.type)

# --------------------------------------------
# Train Model
# --------------------------------------------
train_model(
    model=model,
    num_epochs=num_epochs,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    scaler=scaler,
    train_loader=train_loader,
    val_loader=val_loader,
    patience=patience,
    device=device,
    metric_device=metric_device,
)

In [ ]:
!cp /content/best_model.pth /content/drive/MyDrive/best_model.pth

# **Evaluate the Model**

In [ ]:
import torch

# Load the model's state_dict (replace 'model.pth' with your file name)
# model.load_state_dict(torch.load('/content/drive/MyDrive/best_model_scale_3.pth'))
model.load_state_dict(torch.load('best_model.pth'))
model = model.eval()

In [ ]:
from src.dataset.snow import SnowDataset
from torch.utils.data import DataLoader

dataset_dir = "/content/drive/MyDrive/snow_dataset"  # Replace with your dataset path

# Dataset and DataLoader
batch_size = 10

train_dataset = SnowDataset(
    dataset_dir_path=dataset_dir,
    len_override=30,
    normalize=True,
)
validation_dataset = SnowDataset(
    dataset_dir_path=dataset_dir,
    len_override=30,
    normalize=False,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=1,
)
val_loader = DataLoader(
    validation_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=1,
)

In [ ]:
# Get predictions
images, masks = next(iter(train_loader))
images, masks = images.to(device), masks.to(device)
outputs = model(images)

# Get non-normalized images for evaluation
images_original, _ = next(iter(val_loader))

In [ ]:
import torch.nn as nn

criterion = nn.BCEWithLogitsLoss()  # Binary cross-entropy loss with logits
loss = criterion(outputs, masks)
print(loss)
del loss
del criterion

In [ ]:
from src.training.visualisation import display_attention_weights, display_tensor_mask, display_tensor_image

outputs = torch.sigmoid(outputs)
image_idx = 0

In [ ]:
display_tensor_image(images_original[image_idx])

In [ ]:
display_tensor_mask(masks[image_idx])

In [ ]:
display_tensor_mask(outputs[image_idx] > 0.5)

In [ ]:
del masks, outputs

In [ ]:
for i in range(6):
    display_attention_weights(model, images_original[0], images[0], 8, 'x3', i, True)